# Demo Funcional y Presentación Final
### Fase 4: Deployment y Documentación
**Proyecto I: Introducción a LLMs — Facultad de Ciencias, UNAM — Semestre 2026-2**

---

> **Dónde estamos:** Tenemos modelos evaluados con métricas sólidas. El último paso es empaquetarlos en una demo funcional que cualquier persona pueda usar, y preparar la presentación final.

**Mapa de la sesión:**
```
Parte 1: Del notebook al demo — Gradio en 10 minutos      (15 min)
Parte 2: Demo del clasificador de sentimiento             (15 min)
Parte 3: Demo del chatbot de joyería                      (15 min)
Parte 4: Estructura del reporte técnico                   (10 min)
Parte 5: Checklist de presentación                         (5 min)
```

**Infraestructura:** Colab T4 GPU para los modelos. Gradio genera un enlace público temporal — cualquier persona puede probar el demo desde su teléfono.

In [ ]:
!pip install gradio transformers torch peft sentence-transformers faiss-cpu --quiet

import torch
import gradio as gr
import numpy as np
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo: {device}")
print(f"Gradio version: {gr.__version__}")
print("Listo.")

---
## Parte 1: Del Notebook al Demo — Gradio

Un notebook es para desarrollo. Una demo es para mostrar. La diferencia es la interfaz.

**Gradio** convierte una función de Python en una interfaz web en 3 líneas. Genera un enlace público temporal (72 horas) que funciona en cualquier dispositivo. Es el estándar en la comunidad de ML para demos rápidas — HuggingFace Spaces usa Gradio.

```python
# El patrón completo:
def mi_funcion(input_texto):
    resultado = modelo(input_texto)
    return resultado

demo = gr.Interface(fn=mi_funcion, inputs="text", outputs="text")
demo.launch(share=True)  # share=True genera el enlace público
```

In [ ]:
# Gradio mínimo — para entender la estructura antes de integrar el modelo

def echo(texto):
    return f"Recibí: '{texto}' ({len(texto)} caracteres)"

demo_echo = gr.Interface(
    fn=echo,
    inputs=gr.Textbox(label="Entrada", placeholder="Escribe algo..."),
    outputs=gr.Textbox(label="Salida"),
    title="Demo mínima",
    description="Ejemplo de la estructura de Gradio antes de integrar el modelo real."
)

# Ejecutar inline en Colab (sin share=True primero para verificar que funciona)
demo_echo.launch(inline=True, quiet=True)

---
## Parte 2: Demo — Clasificador de Sentimiento
### *(Proyecto: análisis de sentimiento en reseñas Google Maps)*

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification
)
from peft import PeftModel
import torch.nn.functional as F

# ── Opción A: cargar el modelo fine-tuneado con LoRA ────────────────────────
# Usar esta opción si ya corriste el notebook de LoRA y guardaste el modelo

RUTA_LORA = './modelo_sentimiento_lora'   # ruta del notebook anterior
MODEL_BASE = 'bert-base-multilingual-cased'

import os
if os.path.exists(RUTA_LORA):
    print("Cargando modelo LoRA fine-tuneado...")
    tokenizer_cls = AutoTokenizer.from_pretrained(RUTA_LORA)
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_BASE, num_labels=3
    )
    modelo_cls = PeftModel.from_pretrained(base_model, RUTA_LORA)
    usar_lora = True
    print("Modelo LoRA cargado.")
else:
    # ── Opción B: usar el baseline preentrenado ─────────────────────────────
    # Usar esta opción si no tienes el modelo LoRA guardado
    print("Modelo LoRA no encontrado — usando baseline preentrenado.")
    print("(Ejecuta primero el notebook de LoRA para obtener el modelo fine-tuneado)")
    from transformers import pipeline
    pipe_baseline = pipeline(
        'sentiment-analysis',
        model='nlptown/bert-base-multilingual-uncased-sentiment',
        device=0 if device == 'cuda' else -1
    )
    usar_lora = False
    print("Baseline cargado.")

ETIQUETAS = {0: 'Negativa ⛔', 1: 'Neutra ➖', 2: 'Positiva ✅'}
COLORES   = {0: '#FFEBEE',    1: '#FFF9C4',   2: '#E8F5E9'}

In [ ]:
def clasificar_sentimiento(resena):
    """
    Función que conecta el modelo con la interfaz de Gradio.
    Entrada: texto de una reseña
    Salida: etiqueta de sentimiento + probabilidades
    """
    if not resena or not resena.strip():
        return "Por favor escribe una reseña.", {}

    if usar_lora:
        modelo_cls.eval()
        inputs = tokenizer_cls(
            resena, truncation=True, max_length=128,
            return_tensors='pt'
        )
        with torch.no_grad():
            logits = modelo_cls(**inputs).logits
        probs = F.softmax(logits, dim=-1)[0]
        pred_idx = probs.argmax().item()
        confianza = probs[pred_idx].item()
        distribucion = {
            'Negativa': float(probs[0]),
            'Neutra':   float(probs[1]),
            'Positiva': float(probs[2]),
        }
    else:
        resultado = pipe_baseline(resena[:512])[0]
        estrellas = int(resultado['label'][0])
        pred_idx = 0 if estrellas <= 2 else (1 if estrellas == 3 else 2)
        confianza = resultado['score']
        distribucion = {
            'Negativa': confianza if pred_idx == 0 else (1 - confianza) / 2,
            'Neutra':   confianza if pred_idx == 1 else (1 - confianza) / 2,
            'Positiva': confianza if pred_idx == 2 else (1 - confianza) / 2,
        }

    etiqueta = ETIQUETAS[pred_idx]
    resultado_texto = f"{etiqueta}  (confianza: {confianza:.1%})"
    return resultado_texto, distribucion


# Prueba rápida antes de lanzar la demo
resenas_prueba = [
    "Excelente lugar, la comida estaba deliciosa y el servicio fue muy rápido.",
    "Pésimo servicio, tardaron una hora y la comida llegó fría.",
    "Ni bueno ni malo, un lugar normal.",
]
print("Prueba del clasificador:")
for resena in resenas_prueba:
    etiqueta, probs = clasificar_sentimiento(resena)
    print(f"  {resena[:50]:<50} → {etiqueta}")

In [ ]:
# Demo de Gradio — Clasificador de Sentimiento

ejemplos_sentimiento = [
    "Excelente servicio, el personal fue muy amable y la comida llegó caliente. Definitivamente regresaré.",
    "Pésima experiencia. Tardaron una hora y la comida llegó fría. No lo recomendaría.",
    "El lugar está bien, nada especial pero cumple con lo básico. Precios razonables.",
    "No entiendo por qué tiene tantas reseñas positivas. A mí no me pareció especial.",
    "Amazing food! Best tacos in Mexico City. Will definitely come back!",
]

with gr.Blocks(title="Análisis de Sentimiento — Google Maps") as demo_sentimiento:
    gr.Markdown("""
    # 📍 Analizador de Reseñas de Google Maps
    Clasifica automáticamente el sentimiento de reseñas en español e inglés.
    """)

    with gr.Row():
        with gr.Column(scale=2):
            entrada = gr.Textbox(
                label="Reseña",
                placeholder="Pega aquí la reseña de Google Maps...",
                lines=4
            )
            btn = gr.Button("Analizar sentimiento", variant="primary")

        with gr.Column(scale=1):
            salida_etiqueta = gr.Textbox(label="Sentimiento detectado", interactive=False)
            salida_probs    = gr.Label(label="Distribución de probabilidad", num_top_classes=3)

    gr.Examples(
        examples=ejemplos_sentimiento,
        inputs=entrada,
        label="Ejemplos"
    )

    btn.click(
        fn=clasificar_sentimiento,
        inputs=entrada,
        outputs=[salida_etiqueta, salida_probs]
    )

    gr.Markdown("""
    ---
    **Modelo:** BERT-base-multilingual + LoRA fine-tuning  
    **Proyecto:** Análisis de sentimiento en reseñas — Proyecto I LLMs, FC UNAM 2026-2
    """)

# Lanzar con enlace público para la presentación
demo_sentimiento.launch(share=True, quiet=True)

---
## Parte 3: Demo — Chatbot de Joyería
### *(Proyecto: chatbot para e-commerce artesanal)*

In [ ]:
from transformers import AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import faiss

# Cargar modelos del chatbot RAG
print("Cargando modelos del chatbot...")

embedding_model = SentenceTransformer(
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    device=device
)

GEN_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_ID)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_ID,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    device_map='auto' if device == 'cuda' else None,
)
if device == 'cpu':
    gen_model = gen_model.to('cpu')
gen_model.eval()
print("Modelos cargados.")

In [ ]:
# Reconstruir el vector store del catálogo
# (En producción esto se cargaría desde disco con faiss.read_index)

CHUNKS = [
    "Aretes de plata 925 con turquesa natural. Precio: $650 MXN. Disponibles en azul y verde.",
    "Aretes de cobre con baño de oro y piedra ojo de tigre. Precio: $480 MXN. Stock limitado (3 pares).",
    "Aretes minimalistas de plata lisa forma de luna. Precio: $320 MXN. Siempre disponibles.",
    "Aretes largos de plata con cuentas de obsidiana. Precio: $580 MXN. Disponibles.",
    "Collar de plata con dije de obsidiana tallada a mano. Precio: $1,200 MXN. Cada pieza es única.",
    "Collar de cuarzo rosa en cadena de plata 925. Precio: $980 MXN. Disponible.",
    "Gargantilla de cobre trenzado. Precio: $550 MXN. Disponible en 3 grosores.",
    "Collar choker de plata con piedra labradorita. Precio: $1,100 MXN. Stock limitado (2 piezas).",
    "Pulsera de plata con charms artesanales (colibrí, luna, sol). Precio: $750 MXN. Personalizable.",
    "Brazalete de cobre martillado. Precio: $420 MXN. Talla única ajustable.",
    "Pulsera de plata con piedra de luna (moonstone). Precio: $890 MXN. Disponible.",
    "Envíos a toda la República Mexicana. Costo: $120 MXN. GRATIS en compras mayores a $1,500 MXN. "
    "Tiempo: 3-5 días CDMX, 5-7 días resto del país.",
    "Métodos de pago: tarjeta de crédito/débito, transferencia bancaria (SPEI), PayPal.",
    "Garantía 6 meses en plata 925. Devoluciones: 30 días en producto sin uso y en empaque original.",
    "Pedidos personalizados disponibles. Tiempo adicional: 7-10 días hábiles. Anticipo: 50%.",
    "Materiales: plata 925, cobre artesanal, piedras semipreciosas. No trabajamos oro macizo.",
    "Contacto: Instagram @tallerlunaplata | Email: contacto@tallerlunaplata.com | WhatsApp: 55-1234-5678.",
]

UMBRAL = 0.30

chunk_embeddings = embedding_model.encode(
    CHUNKS, normalize_embeddings=True, show_progress_bar=False
)
index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings.astype(np.float32))

SYSTEM_LUNA = """Eres Luna, la asistente virtual del Taller Luna Plata, tienda de joyería artesanal mexicana.
Personalidad: amable, cálida, apasionada por la joyería. Siempre respondes en español.
Reglas: usa SOLO la información del contexto. No inventes productos ni precios.
Si no tienes la información, di: "No tengo esa información, escríbenos a contacto@tallerlunaplata.com".
Respuestas concisas: máximo 3-4 oraciones."""

print(f"Vector store listo: {index.ntotal} fragmentos indexados.")

In [ ]:
def responder_chatbot(mensaje, historial):
    """
    Función compatible con gr.ChatInterface.
    historial: lista de tuplas (usuario, asistente)
    """
    if not mensaje.strip():
        return ""

    # RETRIEVE
    q_emb = embedding_model.encode(
        [mensaje], normalize_embeddings=True
    ).astype(np.float32)
    scores, indices = index.search(q_emb, 3)
    chunks_relevantes = [
        CHUNKS[idx] for score, idx in zip(scores[0], indices[0])
        if score >= UMBRAL and idx >= 0
    ]

    # AUGMENT
    if chunks_relevantes:
        contexto = "\n".join(f"- {c}" for c in chunks_relevantes)
        system = SYSTEM_LUNA + f"\n\nINFORMACIÓN DISPONIBLE:\n{contexto}"
    else:
        system = SYSTEM_LUNA + "\n\nINFORMACIÓN DISPONIBLE: No se encontró información relevante."

    # Construir historial en formato de mensajes
    mensajes = [{"role": "system", "content": system}]
    for user_msg, asst_msg in (historial or [])[-4:]:  # últimos 4 turnos
        mensajes.append({"role": "user",      "content": user_msg})
        mensajes.append({"role": "assistant", "content": asst_msg})
    mensajes.append({"role": "user", "content": mensaje})

    # GENERATE
    texto = gen_tokenizer.apply_chat_template(
        mensajes, tokenize=False, add_generation_prompt=True
    )
    inputs = gen_tokenizer(texto, return_tensors='pt').to(gen_model.device)
    n_in = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens=180,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=gen_tokenizer.eos_token_id,
        )

    respuesta = gen_tokenizer.decode(
        outputs[0][n_in:], skip_special_tokens=True
    ).strip()
    return respuesta

In [ ]:
# Demo de Gradio — Chatbot de Joyería

ejemplos_chatbot = [
    "¿Qué aretes de plata tienen disponibles?",
    "¿Cuánto cuesta el envío a Monterrey?",
    "Quiero regalarle algo a mi mamá, ¿qué me recomiendas?",
    "¿Hacen pulseras personalizadas con nombres?",
    "¿Tienen anillos de oro?",
]

demo_chatbot = gr.ChatInterface(
    fn=responder_chatbot,
    title="💍 Luna — Asistente de Taller Luna Plata",
    description=(
        "Hola, soy Luna, la asistente virtual del Taller Luna Plata. "
        "Pregúntame sobre nuestros productos, envíos, pagos o pedidos personalizados."
    ),
    examples=ejemplos_chatbot,
    theme=gr.themes.Soft(),
)

demo_chatbot.launch(share=True, quiet=True)


---
## Parte 4: Estructura del Reporte Técnico

El reporte tiene formato de tesis. La siguiente estructura es la recomendada — cada sección tiene un párrafo de qué debe contener.

In [ ]:
estructura_reporte = {
    "1. Introducción": {
        "extensión": "1-2 páginas",
        "contenido": [
            "Motivación: ¿por qué es relevante este problema?",
            "Objetivo general del proyecto",
            "Contribución: ¿qué construiste que no existía antes?",
            "Estructura del reporte",
        ]
    },
    "2. Marco Teórico": {
        "extensión": "3-5 páginas",
        "contenido": [
            "Transformer: arquitectura y mecanismo de atención",
            "Preentrenamiento autosupervisado (MLM o Causal LM)",
            "Fine-tuning eficiente: LoRA — descripción matemática y comparación",
            "RAG (si aplica): recuperación semántica y generación aumentada",
            "Métricas de evaluación relevantes para la tarea",
        ]
    },
    "3. Metodología": {
        "extensión": "3-5 páginas",
        "contenido": [
            "Descripción del dataset: fuente, tamaño, distribución, preprocesamiento",
            "Arquitectura del sistema: diagrama de componentes",
            "Configuración del baseline",
            "Configuración del fine-tuning: hiperparámetros justificados",
            "Protocolo de evaluación: splits, métricas, semilla",
        ]
    },
    "4. Experimentos y Resultados": {
        "extensión": "3-5 páginas",
        "contenido": [
            "Tabla de resultados: baseline vs fine-tuning en test set",
            "Gráficas: curvas de aprendizaje, matriz de confusión, F1 por clase",
            "Análisis de errores con taxonomía y ejemplos",
            "Comparación cualitativa: ejemplos de respuestas correctas e incorrectas",
        ]
    },
    "5. Discusión": {
        "extensión": "1-2 páginas",
        "contenido": [
            "Interpretación de los resultados: ¿qué aprendiste?",
            "Limitaciones del sistema construido",
            "Trabajo futuro: ¿qué harías con más tiempo/recursos?",
        ]
    },
    "6. Conclusiones": {
        "extensión": "0.5-1 página",
        "contenido": [
            "Resumen de contribuciones",
            "Respuesta a los objetivos planteados en la introducción",
        ]
    },
    "7. Referencias": {
        "extensión": "1-2 páginas",
        "contenido": [
            "Vaswani et al. (2017) — Attention is All You Need",
            "Devlin et al. (2019) — BERT",
            "Hu et al. (2021) — LoRA (si usaste fine-tuning)",
            "Lewis et al. (2020) — RAG (si usaste RAG)",
            "Dataset utilizado",
            "Modelos preentrenados de HuggingFace",
        ]
    },
}

print("ESTRUCTURA DEL REPORTE TÉCNICO")
print("=" * 60)
total_pags = 0
for seccion, info in estructura_reporte.items():
    ext = info['extensión']
    print(f"\n{seccion} [{ext}]")
    for item in info['contenido']:
        print(f"  • {item}")

print()
print("Extensión total estimada: 15-25 páginas")
print("Formato: LaTeX o Word, interlineado 1.5, fuente 12pt")

---
## Parte 5: Checklist de Presentación Final

La presentación tiene 20 minutos + 10 de preguntas. La estructura recomendada es:

In [ ]:
slides = [
    ("1. Portada",            "1 min",
     "Título, tu nombre, fecha. Una imagen representativa del proyecto."),

    ("2. Problema y motivación", "2 min",
     "¿Qué problema resuelve tu proyecto? ¿Por qué importa? "
     "Dato concreto que justifique la relevancia."),

    ("3. Arquitectura del sistema", "2 min",
     "Diagrama de los componentes. Una slide, sin texto largo. "
     "Explica el flujo de datos de entrada a salida."),

    ("4. Dataset", "1 min",
     "Tamaño, fuente, distribución de clases (gráfica). "
     "¿Qué fue difícil de conseguir o limpiar?"),

    ("5. Resultados — tabla", "2 min",
     "Tabla comparativa baseline vs tu modelo. "
     "Resalta la métrica principal. Sé honesto con los números."),

    ("6. Resultados — análisis de errores", "2 min",
     "2-3 ejemplos de errores con análisis. "
     "¿Qué tipo de texto confunde al modelo? ¿Por qué?"),

    ("7. Demo en vivo", "5 min",
     "Abre la demo de Gradio. Ten preparados 3-4 ejemplos interesantes. "
     "Incluye un caso exitoso Y un caso donde falla — eso muestra análisis crítico."),

    ("8. Limitaciones y trabajo futuro", "2 min",
     "¿Qué no funciona bien? ¿Qué harías con más tiempo? "
     "Ser crítico de tu propio trabajo es una fortaleza, no una debilidad."),

    ("9. Conclusiones", "1 min",
     "3 bullets máximo. ¿Qué aprendiste? ¿Qué construiste?"),

    ("10. Preguntas",          "10 min",
     "Prepara respuestas para: ¿Por qué elegiste ese modelo? "
     "¿Qué pasaría con más datos? ¿Cómo mejora sobre el baseline?"),
]

print("ESTRUCTURA DE LA PRESENTACIÓN (20 min + 10 preguntas)")
print("=" * 65)
tiempo_total = 0
for slide, tiempo, descripcion in slides:
    print(f"\n{slide} [{tiempo}]")
    print(f"  {descripcion}")

print()
print("Consejos prácticos:")
print("  • Practica la demo antes — ten un video de respaldo por si falla la red")
print("  • No leas las slides — habla con tus propias palabras")
print("  • Si no sabes la respuesta a una pregunta, di 'no lo sé' — es válido")
print("  • El demo en vivo es lo más memorable — dale los 5 minutos")

In [ ]:
# Checklist final de entrega

entregables = {
    "Repositorio de GitHub": [
        "☐ README.md con descripción del proyecto e instrucciones de uso",
        "☐ Notebooks organizados y con salidas guardadas",
        "☐ Carpeta experimentos/ con tarjetas JSON de cada corrida",
        "☐ requirements.txt o environment.yml",
        "☐ Enlace al modelo en HuggingFace Hub (si aplica)",
    ],
    "Reporte técnico": [
        "☐ PDF enviado por Classroom antes de la presentación",
        "☐ Todas las secciones de la estructura recomendada",
        "☐ Tabla de resultados con métricas en test set",
        "☐ Referencias correctamente citadas",
        "☐ Figuras con pie de figura descriptivo",
    ],
    "Presentación": [
        "☐ Demo funcionando (probada el día anterior)",
        "☐ Video de respaldo de la demo",
        "☐ Slides subidas a Classroom",
        "☐ Tiempo de práctica: al menos 2 ensayos completos",
    ],
}

print("CHECKLIST DE ENTREGA FINAL")
print("=" * 50)
for categoria, items in entregables.items():
    print(f"\n{categoria}:")
    for item in items:
        print(f"  {item}")

print()
print("¡Mucho éxito en la presentación!")

---
## Lo que construyeron en este curso

```
Fase 1 — Fundamentos
  Regresión logística → MLP → Transformer desde cero
  BERT y GPT: los dos paradigmas
  Lectura crítica de Attention is All You Need

Fase 2 — Datos y Fine-tuning
  Baseline con modelos preentrenados
  LoRA: ajuste eficiente con <2% de los parámetros
  RAG: retrieval semántico + generación fundamentada

Fase 3 — Evaluación
  Métricas por tipo de tarea (F1-macro, ROUGE, BERTScore)
  Análisis de errores con taxonomía
  Registro reproducible de experimentos

Fase 4 — Deployment
  Demo funcional con Gradio
  Reporte técnico con formato de tesis
  Presentación con demo en vivo
```

Todo esto usando infraestructura gratuita (Colab, Kaggle, HuggingFace) y modelos de código abierto. El mismo pipeline escala a proyectos de investigación y aplicaciones industriales.